# DistilBERT fine-tune

Since this is standalone in kaggle, some functions are replicated

In [1]:
!pip install -q "transformers>=4.45,<5" "datasets>=3.0" emoji onnx onnxruntime accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 682.7 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 53.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 49.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.1 MB/s eta 0:00:00


In [26]:
import numpy as np
from pathlib import Path
import onnxruntime as ort
from collections import Counter
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers import AutoTokenizer
from datasets import load_dataset, Dataset
import json, shutil, re, html, unicodedata, emoji, torch
import transformers, onnxruntime, datasets
from sklearn.utils.class_weight import compute_class_weight
from onnxruntime.quantization import QuantType, quantize_dynamic

In [3]:
device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only"
print("device: ", device)

device:  Tesla T4


# Hyperparameter Configs

In [4]:
BASE_MODEL = "distilbert-base-uncased"
DATASET = "cardiffnlp/tweet_eval"
DATASET_CONFIG = "sentiment"
LABELS = {0: "negative", 1: "neutral", 2: "positive"}

MAX_LENGTH = 96
EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
SEED = 42
FP16 = torch.cuda.is_available()
METRIC_FOR_BEST = "eval_macro_f1"

OUTPUT_DIR = "/kaggle/working"
CONFIG = {
    "base_model": BASE_MODEL,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "seed": SEED,
    "fp16": FP16,
    "metric_for_best_model": METRIC_FOR_BEST,
}
CONFIG

{'base_model': 'distilbert-base-uncased',
 'max_length': 96,
 'epochs': 3,
 'batch_size': 32,
 'learning_rate': 2e-05,
 'weight_decay': 0.01,
 'warmup_ratio': 0.06,
 'seed': 42,
 'fp16': True,
 'metric_for_best_model': 'eval_macro_f1'}

# Cleaning the Data

In [5]:
ESCAPE_RE = re.compile(r"\\u([0-9a-fA-F]{4})")
URL_RE = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"@\w{1,15}\b")
WS_RE = re.compile(r"\s+")


def _escaped_char(match):
    return chr(int(match.group(1), 16))


def expand_escapes(text):
    """Expand escaped unicode and HTML entities until the text stops changing."""
    while True:
        expanded = html.unescape(ESCAPE_RE.sub(_escaped_char, text))
        if expanded == text:
            return text
        text = expanded


def normalize(text):
    """Normalize text while preserving sentiment-bearing case and punctuation."""
    normalized = unicodedata.normalize("NFKC", expand_escapes(text))
    normalized = URL_RE.sub("[url]", normalized)
    normalized = MENTION_RE.sub("[user]", normalized)
    normalized = emoji.demojize(normalized, delimiters=("[", "]"))
    return WS_RE.sub(" ", normalized).strip()


checks = {
    r"today\u002c not perfect": "today, not perfect",
    "Fish &amp; chips": "Fish & chips",
    "Hi @alice https://example.com": "Hi [user] [url]",
    "  \uff37\uff2f\uff37  ": "WOW",
}
for raw, expected in checks.items():
    got = normalize(raw)
    assert got == expected, f"{raw!r} -> {got!r}, expected {expected!r}"
print("cleaner self-checks passed")
for raw in list(checks)[:3]:
    print(f"{raw!r} -> {normalize(raw)!r}")

cleaner self-checks passed
'today\\u002c not perfect' -> 'today, not perfect'
'Fish &amp; chips' -> 'Fish & chips'
'Hi @alice https://example.com' -> 'Hi [user] [url]'


## Load and clean

In [6]:
raw = load_dataset(DATASET, DATASET_CONFIG)
data = {}
for split in ("train", "validation", "test"):
    rows = raw[split]
    data[split] = {
        "text": [normalize(str(t)) for t in rows["text"]],
        "label": [int(v) for v in rows["label"]],
    }
    print(f"{split:11s} {len(data[split]['label']):6,} rows")

print()
for text in data["train"]["text"][:3]:
    print(" ", text[:90])

README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

train       45,615 rows
validation   2,000 rows
test        12,284 rows

  "QT [user] In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwa
  "Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"
  Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back 


## 4. Class weights

In [7]:
train_labels = data["train"]["label"]
counts = Counter(train_labels)
class_weights = compute_class_weight(
    "balanced", classes=np.array(sorted(LABELS)), y=np.array(train_labels)
)
for label_id in sorted(LABELS):
    share = counts[label_id] / len(train_labels) * 100
    name = LABELS[label_id]
    weight = class_weights[label_id]
    print(f"{name:9s} {counts[label_id]:6,}  {share:5.2f}%   weight {weight:.4f}")

CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32)

negative   7,093  15.55%   weight 2.1437
neutral   20,673  45.32%   weight 0.7355
positive  17,849  39.13%   weight 0.8519


## 5. Tokenize

In [8]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def encode(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = {}
for split, rows in data.items():
    dataset = Dataset.from_dict({"text": rows["text"], "labels": rows["label"]})
    tokenized[split] = dataset.map(encode, batched=True, remove_columns=["text"])

lengths = [len(ids) for ids in tokenized["train"]["input_ids"]]
percentiles = [np.percentile(lengths, q) for q in (50, 95, 99)]
print(
    f"wordpiece p50 {percentiles[0]:.0f}  p95 {percentiles[1]:.0f}  "
    f"p99 {percentiles[2]:.0f}  max {max(lengths)}"
)
truncated = sum(n >= MAX_LENGTH for n in lengths) / len(lengths)
print(f"truncated at {MAX_LENGTH}: {truncated:.3%}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/45615 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

wordpiece p50 30  p95 41  p99 47  max 96
truncated at 96: 0.002%


## 6. Weighted trainer

In [9]:
set_seed(SEED)

class WeightedTrainer(Trainer):
    """Applies inverse-frequency class weights inside the loss."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weights = CLASS_WEIGHTS.to(outputs.logits.device)
        loss = F.cross_entropy(outputs.logits, labels, weight=weights)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(prediction):
    predicted = prediction.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(prediction.label_ids, predicted),
        "macro_f1": f1_score(prediction.label_ids, predicted, average="macro"),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=LABELS,
    label2id={name: i for i, name in LABELS.items()},
)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


66,955,779 parameters


## 7. Train

In [10]:
arguments = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    seed=SEED,
    fp16=FP16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model=METRIC_FOR_BEST,
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    report_to=[],
)

trainer = WeightedTrainer(
    model=model,
    args=arguments,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result.metrics)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.646700,0.642634,0.687000,0.674053
2,0.543500,0.645833,0.714000,0.702822
3,0.471400,0.661682,0.717000,0.703589


{'train_runtime': 424.6868, 'train_samples_per_second': 322.226, 'train_steps_per_second': 5.037, 'total_flos': 1701668258074008.0, 'train_loss': 0.5870507995120383, 'epoch': 3.0}


## Reference scores

In [11]:
fp32_scores = {}
for split in ("validation", "test"):
    metrics = trainer.evaluate(tokenized[split], metric_key_prefix=split)
    fp32_scores[split] = {
        "accuracy": float(metrics[f"{split}_accuracy"]),
        "macro_f1": float(metrics[f"{split}_macro_f1"]),
    }
    print(
        f"{split:11s} accuracy {fp32_scores[split]['accuracy']:.4f}   "
        f"macro-F1 {fp32_scores[split]['macro_f1']:.4f}"
    )

validation  accuracy 0.7170   macro-F1 0.7036
test        accuracy 0.6746   macro-F1 0.6755


## Export to ONNX, then quantize to int8

In [27]:
export_dir = Path(OUTPUT_DIR) / "artifacts"
export_dir.mkdir(parents=True, exist_ok=True)
fp32_path = export_dir / "model_fp32.onnx"
int8_path = export_dir / "model_int8.onnx"

model.eval().cpu()
sample = tokenizer(
    "a sample tweet for tracing",
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
)

torch.onnx.export(
    model,
    (sample["input_ids"], sample["attention_mask"]),
    str(fp32_path),
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "sequence"},
        "attention_mask": {0: "batch", 1: "sequence"},
        "logits": {0: "batch"},
    },
    opset_version=17,
    do_constant_folding=True,
    dynamo=False
)
print(f"fp32 {fp32_path.stat().st_size / 1_000_000:.1f} MB")

quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QInt8)
print(f"int8 {int8_path.stat().st_size / 1_000_000:.1f} MB")

/tmp/ipykernel_58/2838762170.py:14: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


fp32 267.9 MB


int8 67.3 MB


## Check the export against PyTorch

In [28]:
session = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
probe = [normalize(t) for t in raw["test"]["text"][:256]]
encoded = tokenizer(
    probe, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="np"
)

onnx_logits = session.run(
    ["logits"],
    {
        "input_ids": encoded["input_ids"].astype("int64"),
        "attention_mask": encoded["attention_mask"].astype("int64"),
    },
)[0]
torch_inputs = tokenizer(
    probe, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt"
)
with torch.no_grad():
    torch_logits = model(**torch_inputs).logits.numpy()

agreement = (onnx_logits.argmax(-1) == torch_logits.argmax(-1)).mean()
print(f"int8 vs fp32 label agreement on {len(probe)} rows: {agreement:.2%}")
print("quantization moves a few borderline rows; macro-F1 is measured locally")

int8 vs fp32 label agreement on 256 rows: 89.84%
quantization moves a few borderline rows; macro-F1 is measured locally


## Package for download

In [29]:
tokenizer.save_pretrained(export_dir / "tokenizer")
fp32_path.unlink()

summary = {
    "model_name": "distilbert",
    "base_model": BASE_MODEL,
    "dataset": f"{DATASET}/{DATASET_CONFIG}",
    "config": CONFIG,
    "fp32_scores": fp32_scores,
    "train_metrics": {
        key: float(value)
        for key, value in train_result.metrics.items()
        if isinstance(value, int | float)
    },
    "log_history": trainer.state.log_history,
    "int8_vs_fp32_label_agreement": float(agreement),
    "versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "onnxruntime": onnxruntime.__version__,
        "emoji": emoji.__version__,
    },
}
(export_dir / "training_summary.json").write_text(json.dumps(summary, indent=2))

archive = shutil.make_archive(f"{OUTPUT_DIR}/distilbert_artifacts", "zip", export_dir)
print(f"{archive}  {Path(archive).stat().st_size / 1_000_000:.1f} MB")
for path in sorted(export_dir.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(export_dir)}  {path.stat().st_size / 1_000_000:.2f} MB")

/kaggle/working/distilbert_artifacts.zip  42.3 MB
  model_int8.onnx  67.35 MB
  tokenizer/special_tokens_map.json  0.00 MB
  tokenizer/tokenizer.json  0.71 MB
  tokenizer/tokenizer_config.json  0.00 MB
  tokenizer/vocab.txt  0.23 MB
  training_summary.json  0.01 MB


Unzip into `model/artifacts/distilbert/` locally and run `make eval`.